# Time Series Forecasting: S&P 500

**Goal:** Understand and forecast fluctuations in a major financial index to support investment and budgeting decisions.

**Workflow:** load data → explore → returns & volatility → stationarity tests → decomposition → ACF/PACF → ARIMA → backtest vs naive baseline → residual diagnostics → conclusions.

> Requires: `pip install yfinance statsmodels seaborn matplotlib pandas numpy`

## 1. Setup and data loading

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
import yfinance as yf
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_theme()

In [ ]:
# Download S&P 500 data
sp500 = yf.download('^GSPC', start='2013-01-01', end='2022-12-31')

# Newer yfinance versions return MultiIndex columns; flatten them
if isinstance(sp500.columns, pd.MultiIndex):
    sp500.columns = sp500.columns.droplevel(1)

print("S&P 500 Dataset:")
print(sp500.head())

print("\nDataset information:")
print(sp500.info())

print("\nSummary Statistics:")
print(sp500['Close'].describe())

print("\nMissing values in each column:")
print(sp500.isnull().sum())

In [ ]:
# Plot the closing price
plt.figure(figsize=(14, 7))
plt.plot(sp500.index, sp500['Close'])
plt.title('S&P 500 Closing Price (2013-2022)')
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.grid(True)
plt.tight_layout()
plt.show()

# Select the closing price as the primary series, on a business-day calendar
ts_data = sp500['Close'].dropna().asfreq('B').ffill()
ts_data.name = 'Close'
print("\nSelected time series shape:", ts_data.shape)

## 2. Trend: moving averages

In [ ]:
roll_mean = ts_data.rolling(window=50).mean()
roll_mean_200 = ts_data.rolling(window=200).mean()

plt.figure(figsize=(14, 6))
plt.plot(ts_data, label='Close', alpha=0.6)
plt.plot(roll_mean, label='50-day MA', linewidth=2)
plt.plot(roll_mean_200, label='200-day MA', linewidth=2)
plt.title('S&P 500 with Moving Averages')
plt.legend()
plt.tight_layout()
plt.show()

## 3. Log returns and volatility

Prices are non-stationary; returns are the quantity we can model more reliably. Rolling volatility shows risk regimes.

In [ ]:
log_returns = np.log(ts_data).diff().dropna()
rolling_vol = log_returns.rolling(21).std() * np.sqrt(252)   # annualized, ~1 month window

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(log_returns, linewidth=0.7)
axes[0].set_title('Daily Log Returns')
axes[1].plot(rolling_vol, color='crimson')
axes[1].set_title('21-Day Rolling Annualized Volatility')
plt.tight_layout()
plt.show()

print("Annualized return: {:.2%}".format(log_returns.mean() * 252))
print("Annualized volatility: {:.2%}".format(log_returns.std() * np.sqrt(252)))

## 4. Stationarity tests (Augmented Dickey-Fuller)

Null hypothesis: the series has a unit root (non-stationary). A p-value below 0.05 suggests stationarity.

In [ ]:
def adf_report(series, name):
    stat, pval, _, _, crit, _ = adfuller(series.dropna(), autolag='AIC')
    print(f"--- ADF test: {name} ---")
    print(f"ADF statistic: {stat:.4f} | p-value: {pval:.4f}")
    for k, v in crit.items():
        print(f"  Critical value ({k}): {v:.4f}")
    print("=> Stationary" if pval < 0.05 else "=> NON-stationary (fail to reject unit root)")
    print()

adf_report(ts_data, 'Close price')
adf_report(np.log(ts_data), 'Log price')
adf_report(log_returns, 'Log returns')

## 5. Decomposition

In [ ]:
decomp = seasonal_decompose(np.log(ts_data), model='additive', period=252)  # ~1 trading year
fig = decomp.plot()
fig.set_size_inches(14, 9)
plt.tight_layout()
plt.show()

## 6. ACF and PACF

Returns: little linear autocorrelation is expected. Squared returns: strong autocorrelation indicates volatility clustering.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(log_returns, lags=40, ax=axes[0])
plot_pacf(log_returns, lags=40, ax=axes[1], method='ywm')
plt.tight_layout()
plt.show()

plot_acf(log_returns**2, lags=40)
plt.title('ACF of Squared Returns (volatility clustering)')
plt.tight_layout()
plt.show()

## 7. Train / test split

Time series must be split chronologically. We hold out the final year (2022).

In [ ]:
log_price = np.log(ts_data)
split_date = '2022-01-01'
train = log_price[log_price.index < split_date]
test = log_price[log_price.index >= split_date]
print(f"Train: {train.index.min().date()} -> {train.index.max().date()} ({len(train)} obs)")
print(f"Test:  {test.index.min().date()} -> {test.index.max().date()} ({len(test)} obs)")

## 8. ARIMA order selection

We use d=1 (the log price needs one difference) and search p, q in 0..3 by AIC.

In [ ]:
results = []
for p in range(0, 4):
    for q in range(0, 4):
        try:
            fit = ARIMA(train, order=(p, 1, q)).fit()
            results.append(((p, 1, q), fit.aic, fit.bic))
        except Exception:
            continue

order_df = pd.DataFrame(results, columns=['order', 'AIC', 'BIC']).sort_values('AIC')
print(order_df.head(5))
best_order = order_df.iloc[0]['order']
print("Best order by AIC:", best_order)

## 9. Fit and forecast the holdout period

In [ ]:
model = ARIMA(train, order=best_order).fit()
print(model.summary())

fc = model.get_forecast(steps=len(test))
fc_mean = np.exp(fc.predicted_mean)
fc_ci = np.exp(fc.conf_int(alpha=0.05))
fc_mean.index = test.index
fc_ci.index = test.index

actual = np.exp(test)
naive = pd.Series(np.exp(train.iloc[-1]), index=test.index)   # flat last-value baseline

plt.figure(figsize=(14, 6))
plt.plot(np.exp(train[-250:]), label='Train (last year)')
plt.plot(actual, label='Actual', color='black')
plt.plot(fc_mean, label=f'ARIMA{best_order} forecast', color='red', linestyle='--')
plt.fill_between(fc_ci.index, fc_ci.iloc[:, 0], fc_ci.iloc[:, 1],
                 color='red', alpha=0.15, label='95% interval')
plt.plot(naive, label='Naive baseline', color='gray', linestyle=':')
plt.title('S&P 500: ARIMA Forecast vs Actual (2022 holdout)')
plt.legend()
plt.tight_layout()
plt.show()

## 10. Evaluation against the naive baseline

In [ ]:
def metrics(y, yhat):
    err = y - yhat
    return {'MAE': np.mean(np.abs(err)),
            'RMSE': np.sqrt(np.mean(err**2)),
            'MAPE %': np.mean(np.abs(err / y)) * 100}

print(pd.DataFrame({'ARIMA': metrics(actual, fc_mean),
                    'Naive': metrics(actual, naive)}).round(2))

coverage = ((actual >= fc_ci.iloc[:, 0]) & (actual <= fc_ci.iloc[:, 1])).mean()
print(f"\n95% interval coverage on test set: {coverage:.1%}")

## 11. Residual diagnostics

Good residuals look like white noise. Significant autocorrelation in *squared* residuals means volatility is not constant (ARCH effects).

In [ ]:
resid = model.resid[1:]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(resid)
axes[0].set_title('Residuals')
sns.histplot(resid, kde=True, ax=axes[1])
axes[1].set_title('Residual distribution')
plot_acf(resid, lags=30, ax=axes[2])
axes[2].set_title('Residual ACF')
plt.tight_layout()
plt.show()

print("Ljung-Box on residuals:")
print(acorr_ljungbox(resid, lags=[10, 20], return_df=True))
print("\nLjung-Box on squared residuals (ARCH check):")
print(acorr_ljungbox(resid**2, lags=[10, 20], return_df=True))

## 12. Findings and recommendations

*Edit this section after running the notebook and reviewing your own outputs.*

**Observations**
- **Stationarity:** Prices and log prices are expected to be non-stationary; log returns stationary. This justifies differencing (d=1).
- **Autocorrelation:** Returns show little linear structure, so predictability of direction is limited.
- **Volatility clustering:** Squared returns and squared residuals show strong autocorrelation, so risk comes in regimes (e.g., early 2020, 2022).
- **Seasonality:** The seasonal component is weak relative to trend and should not drive decisions.
- **Backtest:** Compare ARIMA's MAE/RMSE/MAPE with the naive baseline. The 2022 holdout is a bear market following a long uptrend, so a trend-following model is expected to over-forecast.
- **Interval coverage:** If the 95% interval covers well under 95% of actuals, the model understates uncertainty.

**Recommendations for planning and budgeting**
1. Use forecasts as scenario ranges, not point predictions.
2. Always benchmark against the naive baseline before trusting a model.
3. Add a volatility model (GARCH, via the `arch` package) for more realistic uncertainty bands.
4. Re-fit regularly and use rolling-origin (walk-forward) validation rather than a single split.
5. Combine statistical forecasts with macro and domain judgment, since structural breaks are not predictable from price history alone.